In [ ]:
# =======================================================
# Ejemplo completo: Forecasting de Volúmenes de Ventas de Seguros en Canales Bancarios
# Usamos NeuralForecast (NHITS) para forecasting mensual de pólizas vendidas con covariables externas
# Simulamos escenarios base/stress + Monte Carlo para impacto en ingresos, cartera de créditos y reservas de seguros
# Monte Carlo para distribución de impactos, clasificación de riesgo cliente (bajo/medio/alto con probabilidades ajustadas),
# métricas completas (forecast + clasificación), análisis de sensibilidad y stress test
# =======================================================
# Requisitos (instala si no los tienes):
# pip install pandas numpy scikit-learn matplotlib seaborn neuralforecast shap

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (classification_report, confusion_matrix, roc_auc_score,
                             roc_curve, average_precision_score)
from neuralforecast import NeuralForecast
from neuralforecast.models import NHITS
from neuralforecast.losses.pytorch import MAE
from neuralforecast.losses.numpy import mae, rmse
import shap
import warnings
warnings.filterwarnings('ignore')

# =======================================================
# 1. Generación de dataset sintético profesional desde cero
# =======================================================
# Simulamos datos mensuales desde 2015 hasta 2025 (132 meses)
# 16 variables: 8 macroeconómicas (covariables externas) + 8 bancarias/ventas agregadas
# Target principal: 'policies_sold' (pólizas vendidas mensuales en canales bancarios)
np.random.seed(42)
dates = pd.date_range('2015-01-01', '2025-12-01', freq='M')
n_periods = len(dates)

# Macro covariables externas (8)
macro_data = {
    'gdp_growth': np.cumsum(np.random.normal(0.2, 1, n_periods)) + 3,  # Crecimiento GDP (%)
    'unemployment_rate': np.abs(np.cumsum(np.random.normal(0, 0.3, n_periods)) + 5),  # Desempleo (%)
    'inflation_rate': np.abs(np.random.normal(2, 1, n_periods)),  # Inflación (%)
    'interest_rate': np.abs(np.random.normal(3, 1.5, n_periods)).clip(0.5, 10),  # Tasa interés (%)
    'consumer_confidence': np.random.normal(100, 15, n_periods).clip(50, 150),  # Confianza consumidor
    'digital_adoption_index': np.cumsum(np.random.normal(0.5, 2, n_periods)) + 50,  # Índice adopción digital
    'bank_branch_density': np.random.normal(20, 5, n_periods).clip(10, 40),  # Densidad sucursales
    'marketing_spend': np.random.normal(500000, 100000, n_periods).clip(200000, 1000000)  # Gasto marketing ($)
}

# Bancarias/Ventas agregadas (8)
bank_sales_data = {
    'active_bank_clients': np.cumsum(np.random.normal(1000, 200, n_periods)) + 500000,  # Clientes activos
    'cross_sell_rate': np.random.normal(15, 3, n_periods).clip(5, 30),  # Tasa cross-sell (%)
    'policies_sold': np.abs(np.random.normal(5000, 1500, n_periods) + 
                           200 * macro_data['gdp_growth'] + 
                           100 * macro_data['consumer_confidence'] + 
                           50 * macro_data['digital_adoption_index'] + 
                           0.001 * macro_data['marketing_spend']).astype(int),  # Pólizas vendidas (target)
    'average_premium': np.random.normal(1200, 300, n_periods).clip(500, 3000),  # Premium promedio ($)
    'claim_rate': np.abs(np.random.normal(8, 2, n_periods)),  # Tasa siniestros (%)
    'customer_acquisition_cost': np.random.normal(200, 50, n_periods).clip(100, 400),  # Costo adquisición ($)
    'retention_rate': np.random.normal(85, 5, n_periods).clip(70, 95),  # Tasa retención (%)
    'net_promoter_score': np.random.normal(40, 10, n_periods).clip(10, 70)  # NPS
}

# Crear DataFrame agregado
df_aggregate = pd.DataFrame({**macro_data, **bank_sales_data})
df_aggregate['ds'] = dates

print(f"Dataset agregado creado: {df_aggregate.shape[0]} periodos, {df_aggregate.shape[1]-1} variables")

# Simular datos a nivel cliente (10,000 clientes) para clasificación de propensión individual
n_clients = 10000
client_data = {
    'client_id': range(1, n_clients + 1),
    'age': np.random.randint(25, 70, n_clients),
    'income': np.random.normal(60000, 25000, n_clients).clip(20000, 150000).astype(int),
    'credit_score': np.random.normal(700, 100, n_clients).clip(300, 850).astype(int),
    'debt_to_income_ratio': np.round(np.random.uniform(0.1, 0.6, n_clients), 2),
    'num_bank_products': np.random.randint(1, 6, n_clients),
    'tenure_years': np.random.randint(1, 20, n_clients),
    'digital_usage_score': np.random.randint(1, 10, n_clients),
    'satisfaction_score': np.random.randint(1, 10, n_clients),
    'previous_insurance': np.random.choice([0, 1], n_clients, p=[0.7, 0.3]),
    'family_size': np.random.randint(1, 6, n_clients),
    'employment_stability': np.random.choice(['Stable', 'Unstable'], n_clients),
    'region': np.random.choice(['Urban', 'Suburban', 'Rural'], n_clients),
    'marketing_exposure': np.random.randint(0, 5, n_clients),
    'complaints': np.random.randint(0, 3, n_clients),
    'life_events': np.random.choice(['None', 'Marriage', 'Child Birth', 'Home Purchase'], n_clients)
}

df_clients = pd.DataFrame(client_data)

# Generar 'buy_insurance' (propensión a comprar seguro)
buy_prob_client = (
    0.20 + 
    0.15 * (df_clients['income'] > 80000) + 
    0.12 * (df_clients['credit_score'] > 700) + 
    0.10 * (df_clients['satisfaction_score'] > 7) + 
    0.08 * (df_clients['num_bank_products'] > 3) + 
    0.07 * (df_clients['digital_usage_score'] > 7) + 
    0.05 * (df_clients['previous_insurance'] == 1) + 
    -0.10 * (df_clients['complaints'] > 0)
)
buy_prob_client = buy_prob_client.clip(0, 0.95)
df_clients['buy_insurance'] = np.random.binomial(1, buy_prob_client)

print(f"Dataset clientes creado: {df_clients.shape[0]} clientes, {df_clients.shape[1]} variables")

# =======================================================
# 2. Exploratory Data Analysis (EDA)
# =======================================================
print("\n=== EDA Agregado ===")
print(df_aggregate.describe())

plt.figure(figsize=(14,8))
plt.plot(df_aggregate['ds'], df_aggregate['policies_sold'], label='Pólizas Vendidas')
plt.plot(df_aggregate['ds'], df_aggregate['consumer_confidence'], label='Confianza Consumidor', alpha=0.7)
plt.plot(df_aggregate['ds'], df_aggregate['gdp_growth'], label='GDP Growth', alpha=0.7)
plt.title('Ventas de Pólizas y Covariables Externas')
plt.legend()
plt.show()

corr_agg = df_aggregate.drop('ds', axis=1).corr()
plt.figure(figsize=(12,10))
sns.heatmap(corr_agg, annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlación Variables Agregadas')
plt.show()

print("\n=== EDA Clientes ===")
plt.figure(figsize=(8,5))
sns.countplot(x='buy_insurance', data=df_clients)
plt.title('Distribución Compra Seguro Clientes')
plt.show()

# =======================================================
# 3. Forecasting con NeuralForecast (policies_sold con covariables externas)
# =======================================================
# Seleccionar covariables externas clave
exog_cols = ['consumer_confidence', 'gdp_growth', 'marketing_spend', 'unemployment_rate']

# Preparar df para NeuralForecast
df_ts = pd.DataFrame({
    'unique_id': ['policies_sold'] * n_periods,
    'ds': dates,
    'y': df_aggregate['policies_sold']
})
for col in exog_cols:
    df_ts[col] = df_aggregate[col]

train_df = df_ts[df_ts['ds'] < '2024-01-01']

# Future exog base (último valor constante)
future_dates = pd.date_range('2024-01-01', '2025-12-01', freq='M')
futr_df_base = pd.DataFrame({
    'unique_id': ['policies_sold'] * len(future_dates),
    'ds': future_dates
})
for col in exog_cols:
    futr_df_base[col] = df_aggregate[col].iloc[-1]

horizon = len(future_dates)

models = [
    NHITS(h=horizon,
          input_size=36,
          futr_exog_list=exog_cols,
          loss=MAE(),
          max_steps=500,
          random_seed=42)
]

nf = NeuralForecast(models=models, freq='M')
nf.fit(df=train_df)

forecast_base = nf.predict(futr_df=futr_df_base)

# =======================================================
# 4. Stress Scenario y Monte Carlo Simulation
# =======================================================
# Stress: caída confianza -20%, desempleo +3%, marketing -30%
futr_df_stress = futr_df_base.copy()
futr_df_stress['consumer_confidence'] *= 0.8
futr_df_stress['unemployment_rate'] *= 1.3
futr_df_stress['marketing_spend'] *= 0.7

forecast_stress = nf.predict(futr_df=futr_df_stress)

# Monte Carlo
n_mc = 1000
mc_sales = []
mc_credit_impact = []
mc_reserves = []

average_premium = df_aggregate['average_premium'].mean()
portfolio_value = 100000000  # Cartera créditos base
reserve_ratio = 0.1  # 10% reservas sobre ingresos seguros

for _ in range(n_mc):
    # Ruido en forecast
    noise = np.random.normal(1, 0.15, size=horizon)
    sales_stress = forecast_stress['NHITS'] * noise
    total_sales = sales_stress.mean()
    
    revenue = total_sales * average_premium * 12  # Anualizado
    reserves = revenue * reserve_ratio
    
    # Impacto cartera créditos (menos ventas seguros -> menos cross-sell)
    credit_impact = -0.3 * (total_sales - forecast_base['NHITS'].mean()) * average_premium
    
    mc_sales.append(total_sales)
    mc_credit_impact.append(credit_impact)
    mc_reserves.append(reserves)

mc_sales = np.array(mc_sales)
mc_credit_impact = np.array(mc_credit_impact)
mc_reserves = np.array(mc_reserves)

print("\n=== Resultados Monte Carlo Stress ===")
print(f"Ventas Pólizas (media): {mc_sales.mean():,.0f}")
print(f"Impacto Cartera Créditos (media): ${mc_credit_impact.mean():,.0f}")
print(f"Reservas Seguros (media): ${mc_reserves.mean():,.0f}")
print(f"VaR 95% Reservas: ${np.percentile(mc_reserves, 95):,.0f}")

# =======================================================
# 5. Clasificación de Riesgo Individual (Clientes - baja propensión = alto riesgo)
# =======================================================
cat_cols_client = df_clients.select_dtypes(include='object').columns
for col in cat_cols_client:
    le = LabelEncoder()
    df_clients[col] = le.fit_transform(df_clients[col])

X_client = df_clients.drop(['client_id', 'buy_insurance'], axis=1)
y_client = df_clients['buy_insurance']

scaler_client = StandardScaler()
X_client_scaled = scaler_client.fit_transform(X_client)

model_prop = LogisticRegression(random_state=42)
model_prop.fit(X_client_scaled, y_client)

prob_prop_base = model_prop.predict_proba(X_client_scaled)[:, 1]

# Ajuste stress (menor ventas agregadas reducen propensión individual)
stress_factor_prop = forecast_stress['NHITS'].mean() / forecast_base['NHITS'].mean()
prob_prop_stress = np.clip(prob_prop_base * stress_factor_prop, 0, 1)

def classify_prop_risk(prob):
    if prob > 0.7:
        return 'Bajo'  # Alta propensión = bajo riesgo no venta
    elif prob > 0.3:
        return 'Medio'
    else:
        return 'Alto'  # Baja propensión = alto riesgo

df_clients['prob_prop_stress'] = prob_prop_stress
df_clients['risk_level'] = df_clients['prob_prop_stress'].apply(classify_prop_risk)

print("\n=== Ejemplos riesgo clientes bajo stress (primeros 10) ===")
print(df_clients[['prob_prop_stress', 'risk_level']].head(10))

print("\nDistribución riesgo clientes bajo stress:")
print(df_clients['risk_level'].value_counts(normalize=True) * 100)

# =======================================================
# 6. Métricas de evaluación
# =======================================================
# Forecast (usando test real)
test_actual = df_aggregate[df_aggregate['ds'] >= '2024-01-01']['policies_sold'].values
forecast_test = forecast_base['NHITS'].values
print("\n=== Métricas Forecast Base ===")
print(f"MAE: {mae(test_actual, forecast_test):.4f}")
print(f"RMSE: {rmse(test_actual, forecast_test):.4f}")

# Clasificación propensión cliente
y_pred_prop = model_prop.predict(X_client_scaled)
print("\n=== Classification Report Propensión Cliente ===")
print(classification_report(y_client, y_pred_prop))
print(f"AUC-ROC Propensión: {roc_auc_score(y_client, prob_prop_base):.4f}")

# =======================================================
# 7. Análisis de sensibilidad y Stress Test
# =======================================================
shock_levels = np.linspace(-30, 10, 11)  # Shock confianza consumidor -30% a +10%
impact_sales = []
base_sales = forecast_base['NHITS'].mean()

for shock in shock_levels:
    stressed_sales = base_sales * (1 + shock/100)
    impact_sales.append(stressed_sales)

plt.figure(figsize=(10,6))
plt.plot(shock_levels, impact_sales)
plt.title('Sensibilidad: Impacto Ventas vs Shock Confianza Consumidor')
plt.xlabel('Shock Confianza (%)')
plt.ylabel('Ventas Pólizas Esperadas')
plt.show()

print("\n=== Stress Test Resumen ===")
print(f"Escenario Base - Ventas Pólizas: {base_sales:,.0f}")
print(f"Escenario Stress - Ventas (media MC): {mc_sales.mean():,.0f}")

# =======================================================
# 8. Análisis de sensibilidad con SHAP (para propensión cliente)
# =======================================================
explainer_prop = shap.Explainer(model_prop, X_client_scaled[:500])
shap_values_prop = explainer_prop(X_client_scaled[:200])

shap.summary_plot(shap_values_prop, X_client_scaled[:200], feature_names=X_client.columns)
plt.title('Importancia Global para Propensión Compra Seguro (SHAP)')
plt.show()

high_risk_client_idx = df_clients[df_clients['risk_level'] == 'Alto'].index[0]
shap.force_plot(explainer_prop.expected_value, shap_values_prop.values[high_risk_client_idx % 200], 
                X_client_scaled[high_risk_client_idx], feature_names=X_client.columns, matplotlib=True)
plt.show()

print("\n¡Análisis completo terminado!")
print("NeuralForecast para ventas con covariables externas (confianza, GDP, marketing, desempleo).")
print("Monte Carlo para incertidumbre en ventas, impacto créditos y reservas.")
print("Riesgo cliente = baja propensión bajo stress.")
print("Sensibilidad muestra impacto de confianza en ventas.")